# Week 2 — 주(state)별 약속 배송일은 실제와 맞는가

**Business Analytics · 산업공학과 4학년 · 2026 · Olist 브라질 이커머스 실데이터**

온라인 쇼핑몰은 주문할 때 "며칠까지 도착한다"는 날짜를 안내한다. 짧은 약속은 구매를 유도할 수 있지만 배송이 그만큼 빨라지지 않으면 지연이 늘어난다. 긴 약속은 지연을 줄일 수 있지만 고객이 구매를 포기할 수도 있다. 이 장은 브라질 쇼핑몰 Olist의 주문 99,441건으로 실제 배송에 며칠이 걸렸고 안내한 날짜를 얼마나 지켰는지 세고, 지금의 안내 날짜가 배송지마다 실제와 맞는지, 어느 지역의 안내를 재검토할지 판단한다.

> **실습본이다.** 첫 셀(데이터 준비)을 빼고 코드 셀이 전부 비어 있다. 각 셀의 주석이 무엇을
> 할지 알려 주고, 바로 위 지시문이 그 이유를 설명한다. 수업 중 교수와 함께 한 셀씩 타이핑해
> 채운다. 코드가 채워진 판은 따로 배포하지 않으므로, 수업이 끝나면
> **"파일 → Drive에 사본 저장"** 으로 직접 채운 노트북을 각자 보관한다.

> **120분 수업 운영안.** 데이터 준비와 읽기 15분 → 문제·계획 10분 → 조인 10분 → 날짜·월별 집계 25분 → 휴식 5분 → 주별 비교 10분 → 분위수 후보 비교 20분 → 조정표·해석 실습 20분 → 마무리 5분. 마지막 20분에는 9-3의 재표집 결과 시연과 8분 해석 실습을 포함한다. 재표집 코드 작성은 수업 후 확장 실습으로 둔다. pandas의 열 선택과 조건 필터, 파이썬 목록·사전·반복문을 배운 학생을 기준으로 한 시간 배분이다.

## 1. 데이터

Olist는 브라질의 온라인 쇼핑몰이다. 판매자가 Olist에 상품을 올리고, 고객이 주문하면 판매자가 상품을 포장해 물류사에 넘기고, 물류사가 고객에게 배송한다. 고객은 주문할 때 "며칠까지 도착한다"는 예상 배송일을 안내받는다. 배송에 며칠이 걸리는지는 판매자와 물류사에 달려 있지만, 며칠이라고 안내할지는 Olist가 정한다. 이 장의 데이터는 그 안내가 실제와 얼마나 맞았는지를 볼 수 있는 주문 기록이다.

Olist가 2018년에 익명화해 공개한 실제 거래 기록으로, 2016년 9월부터 2018년 10월까지의 주문 99,441건이다. 표가 둘이다. 하나는 주문 표로 주문 한 건이 한 행이고, 다른 하나는 고객 표로 주문마다 배송지가 적혀 있다. 배송지가 주문 표에 없기 때문에 지역마다 배송이 얼마나 다른지 보려면 두 표를 합쳐야 한다.

주문 표에는 주문이 각 단계를 지난 시각이 있다. 구매한 시각, 결제가 승인된 시각, 판매자가 물류사에 넘긴 시각, 고객이 받은 시각이다. 시각은 주문이 그 단계를 지날 때 시스템이 기록하고, 그 단계를 지나지 않았거나 기록이 누락되면 시각이 비어 있을 수 있다. 그리고 주문할 때 고객에게 안내한 예상 배송일이 있다. 구매한 날부터 고객이 받은 날까지가 실제 배송 소요일이고, 구매한 날부터 예상 배송일까지가 약속 소요일이다. 고객이 받은 날이 예상 배송일보다 늦으면 그 주문은 지연이다. 두 날짜가 주문마다 나란히 있으니 소요일과 지연을 주문마다 셀 수 있다.

| 주문 표 열 | 뜻 |
|---|---|
| `order_id` | 주문 번호 |
| `customer_id` | 고객 표와 잇는 번호. 주문마다 다르다 |
| `order_status` | 주문 상태. 배송 완료(delivered), 취소(canceled), 배송 중(shipped) 등 |
| `order_purchase_timestamp` | 구매 시각 |
| `order_approved_at` | 결제 승인 시각 |
| `order_delivered_carrier_date` | 판매자가 물류사에 넘긴 시각 |
| `order_delivered_customer_date` | 고객이 받은 시각 |
| `order_estimated_delivery_date` | 고객에게 안내한 예상 배송일. 시각 없이 날짜만 있다 |

| 고객 표 열 | 뜻 |
|---|---|
| `customer_id` | 주문 표와 잇는 번호 |
| `customer_unique_id` | 사람을 구별하는 번호. 같은 사람이 여러 번 주문하면 같은 값 |
| `customer_state` | 배송지의 주(state). 브라질의 주와 연방구를 구별하는 27개 지역 코드. SP 상파울루, RJ 리우데자네이루, MG 미나스제라이스, BA 바이아, AM 아마조나스 |
| `customer_city`, `customer_zip_code_prefix` | 배송지의 도시와 우편번호 앞자리 |

기록에 없는 것도 있다. Olist가 예상 배송일을 어떤 규칙으로 정했는지, 약속이 길면 주문이 얼마나 줄고 지연이 나면 고객이 얼마나 떠나는지는 없다. 이 장은 기록에 있는 소요일과 지연으로 답할 수 있는 것까지 정한다.

데이터 확보(다운로드·캐시)는 `balab.load`가 처리하고, 주문 표와 고객 표를 돌려준다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
olist = load("olist")
orders, customers = olist["orders"], olist["customers"]

## 2. 데이터 탐색

이 장에서 셀 것은 주문마다의 실제 소요일, 약속 소요일, 지연이다. 그러려면 한 행이 무엇인지, 시각 열이 계산할 수 있는 형인지, 아직 배송이 끝나지 않은 주문이 얼마나 되는지, 기록이 어느 기간인지 확인해야 한다. 고객 표가 주문마다 한 행인지도 확인해야 두 표를 합쳤을 때 주문 수가 변하지 않는다.

**2-1. 주문 한 건에 어떤 정보가 기록되어 있는지 확인해 보자.**

주문 표를 출력해 열 이름과 시각의 표기, 전체 행 수를 읽는다. 노트북에서는 표를 담은 변수 이름을 코드 셀의 마지막 줄에 쓰면 표가 표시된다. 주문 표의 이름은 `orders`다.

In [2]:
# 2-1. 데이터프레임을 이름만 쳐서 출력한다
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


> 99,441행 8열이다. 구매 시각은 `2017-10-02 10:56:33` 꼴로 표시된다.

**2-2. 배송에 걸린 시간을 계산할 수 있는 자료형인지 확인해 보자.**

화면에 날짜처럼 보여도 문자열로 저장되어 있으면 두 시각을 바로 뺄 수 없다.

주문 표의 `dtypes`를 보면 열마다 자료형이 나온다. 구매 시각과 고객 수령 시각이 어떤 형으로 읽혔는지 확인한다.

In [3]:
# 2-2. 열마다 자료형 (dtypes)
orders.dtypes


,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


> 모든 열의 자료형이 `object`다. 이 데이터의 시각 열은 문자열로 읽혔으므로 날짜형으로 바꾼 뒤 계산한다. `object`라는 자료형 자체가 항상 문자열만 뜻하는 것은 아니다.

**2-3. 배송에 걸린 시간을 알 수 없는 주문이 얼마나 있는지 확인해 보자.**

고객이 받은 시각이 비어 있으면 실제 소요일을 잴 수 없다.

`isna`는 빈 값인 곳을 참으로 표시한다. 여기에 `sum`을 적용하면 참을 1로 세어 열별 빈칸 수를 구한다. 고객 수령 시각 열의 결과를 읽는다.

In [4]:
# 2-3. 열별 빈 칸 수 (isna, sum)
orders.isna().sum(axis=0) # False 0, true 1, 행단위로 합쳐서 열별 합계가 나왔다.
# 열별 결측치의 갯수

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


> 고객이 받은 시각이 2,965건 비어 있다.

**2-4. 아직 배송이 끝나지 않았거나 취소된 주문이 얼마나 있는지 세어 보자.**

받은 시각이 없는 주문을 이해하고 분석 대상을 정하려면 주문 상태도 함께 봐야 한다.

주문 상태가 담긴 `order_status` 열에 `value_counts`를 적용하면 상태별 건수가 나온다.

In [5]:
# 2-4. 주문 상태의 값별 개수 (value_counts)
orders["order_status"].unique() # 해당 열이 가진 고유값
x = orders["order_status"].value_counts() # 해당 열이 가진 고유값 별 몇개
x[1:] # 전체 주문상태 갯수 중 첫번쩨 데이터 제외하고 나머지
x[1:].sum()

np.int64(2963)

> 배송 완료 96,478건, 나머지 2,963건은 배송 중·취소 등이다. 받은 시각의 빈 칸은 2,965건이다.

상태별 건수와 빈 칸의 총수만으로는 어느 주문의 기록이 빠졌는지 알 수 없다. 실제 소요일과 지연은 배송 완료 상태이면서 받은 시각이 있는 주문으로 센다. 따라서 이후 지연율은 취소·미배송 주문을 포함한 전체 주문의 지연율이 아니다.

**2-5. 이 기록이 언제부터 언제까지의 주문인지 확인해 보자.**

분석 결과가 어느 기간을 설명하는지 알아야 한다. 구매 시각 열에서 `min`으로 가장 이른 값, `max`로 가장 늦은 값을 구한다. 현재는 문자열이지만 연·월·일 순서의 같은 형식으로 기록되어 있어 이 순서로 기간을 확인할 수 있다.

In [6]:
# 2-5. 구매 시각의 최소와 최대 (min, max)
orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

('2016-09-04 21:15:19', '2018-10-17 17:30:18')

> 2016년 9월 4일부터 2018년 10월 17일까지 약 2년이다.

**2-6. 주문에 배송지를 붙였을 때 주문이 여러 행으로 늘어날 가능성이 있는지 확인해 보자.**

고객 표에서 같은 연결 번호가 여러 번 나오면 한 주문에 여러 행이 붙을 수 있다.

고객 표의 전체 행 수와 `customer_id`의 서로 다른 값 수를 비교한다. 행 수는 `len`, 서로 다른 값 수는 `nunique`로 구한다.

In [7]:
# 2-6. 고객 표의 행 수와 customer_id의 서로 다른 값 수를 나란히 확인한다 (len, nunique)
len(customers), customers["customer_id"].nunique()

(99441, 99441)

> 고객 표의 행 수와 연결 번호의 서로 다른 값 수가 모두 99,441이다.

고객 표의 연결 번호가 중복되지 않으므로 주문마다 배송지를 하나씩 붙일 수 있다.

## 3. 풀고자 하는 문제

Olist가 정하는 것은 고객에게 안내하는 예상 배송일이다. 약속을 앞당기면 고객에게 매력적일 수 있지만 지연이 늘 수 있고, 늘리면 지연은 줄어도 구매를 포기하는 고객이 생길 수 있다. 지금 이 날짜를 어떤 규칙으로 정하는지는 기록에 없다. 브라질은 넓어 배송지에 따라 걸리는 날이 크게 다를 수 있다. 안내가 지역마다 다른지, 다르다면 실제 걸리는 날과 맞는지는 기록으로 확인할 수 있다.

> ### 이번 주의 의사결정 질문
> **"지금의 주(state)별 약속 배송일은 실제 배송과 맞는가. 어느 주의 약속을 며칠 고쳐야 하는가?"**

이 기록으로 과거 약속을 평가하고 조정 후보를 만들 수 있다. 배송 완료 주문마다 실제 소요일과 약속 소요일이 있으니, 주마다 지금 약속이 며칠이고 실제로는 며칠 걸리며 어느 비율로 늦는지 셀 수 있다. 어느 주의 약속을 며칠로 고칠지는 그 주의 실제 소요일이 어느 값 이하에 어느 비율이 드는지를 보고 정할 수 있다. 그렇게 정한 약속을 지금 약속과 나란히 놓으면 어느 주는 며칠 앞당기고 어느 주는 며칠 늘려야 하는지가 나온다. 약속이 길어질 때 주문이 얼마나 줄어드는지는 기록에 없으므로, 이 장은 분석 기간의 전체 지연율과 비슷한 수준을 유지한다는 조건을 두고 정하되 그 조건이 무엇을 바꾸는지도 같이 본다.

배송 약속을 관리하는 담당자가 지역별 재검토 대상을 고를 때 쓰는 분석이다. 여기서 “지금 약속”은 분석 기간에 실제 안내된 약속을 뜻한다. 가장 최근에 운영 중인 규칙을 재현한 값은 아니다. 같은 주문으로 후보를 만들고 평가하므로, 결과는 과거 기록에 적용한 비교이며 앞으로의 성과를 검증한 값은 아니다.

이 장은 주문에 배송지를 붙인 뒤(5절), 소요일과 지연율을 계산하고(6절), 주별 차이를 확인한다(7절). 이어 분위수로 약속 후보를 만들고(8절), 조정표와 검토 조건을 정리한다(9절).

이 장에 필요한 분포·평균과 분위수·서비스 수준·표본의 개념은 **[2주차 배경지식 — 배송 소요일의 분포와 약속](https://balab-pknu.github.io/bizanalytics/theory/week02.html)** 문서에 있다.

## 4. 분석 계획

**질문.** 지금의 주(state)별 약속 배송일은 실제 배송과 맞는가. 어느 주의 약속을 며칠 고쳐야 하는가.

### 💬 먼저 답해 본다

1. 지연을 어느 두 열로 정하겠는가.
2. 약속을 정할 때 2년 전체를 쓰겠는가, 최근 몇 달을 쓰겠는가.
3. 어느 주의 약속일을 며칠로 잡을지 정하려면 실제 소요일 분포에서 무엇을 보겠는가.

계산에 쓸 기준은 다음과 같다.

| 항목 | 정의 |
|---|---|
| 실제 소요일 | 고객이 받은 날 − 구매한 날. 배송 완료 주문마다 하나 |
| 약속 소요일 | 예상 배송일 − 구매한 날 |
| 지연 | 고객이 받은 날이 예상 배송일보다 늦은 주문. 지연율은 배송 완료이며 수령 시각이 있는 주문 중 지연의 비율 |
| 기간 | 약속을 정하는 데 쓰는 주문의 구매 기간. 6절에서 소요일과 지연율이 시간에 따라 어떻게 변했는지 보고 정한다 |
| 분위수 | 정렬한 소요일에서 정한 비율에 해당하는 위치의 값. 0.90분위수는 90분위수라고도 부른다. 같은 날짜에 받은 주문이 많거나 두 값 사이를 비례에 따라 채우는 보간을 쓰면 실제 약속 준수 비율은 정확히 0.90이 아닐 수 있다 |

### 계획

| 계획 | 무엇을 하는가 | 왜 필요한가 | 절 |
|---|---|---|---|
| ① | 주문 표에 고객 표의 배송지를 붙인다 | 배송지가 주문 표에 없다. 붙인 뒤 주문 수가 그대로인지 확인한다 | 5절 |
| ② | 배송 완료 주문마다 실제 소요일과 약속 소요일을 계산해 지금 약속의 지연율을 재고, 월별 변화를 본 뒤 약속을 정할 기간을 고른다 | 2년 사이 약속과 배송이 바뀌었다면 2년을 합쳐 정한 약속은 지금에 맞지 않는다 | 6절 |
| ③ | 주(state)마다 주문 수, 실제 소요일 중앙값, 지금 약속 소요일, 지연율을 본다 | 지금 약속이 주마다 다른지, 지연율이 주마다 다른지가 드러난다 | 7절 |
| ④ | 주마다 실제 소요일의 분위수로 약속일을 정해 보고, 기존 전체 지연율과 가까운 후보 분위수를 고른 뒤, 전국 하나의 값으로 두면 주마다 어떻게 되는지 본다 | 약속일을 정하는 규칙과 그 뜻이 정해진다 | 8절 |
| ⑤ | 주마다 지금 약속과 새 약속, 지금 지연율과 새 지연율을 나란히 놓고, 주문이 적은 주와 기간을 바꿨을 때를 확인해 권고한다 | 재검토할 지역과 조정 후보를 정리하고, 적용 전에 확인할 조건을 구분한다 | 9절 |

## 5. 계획 ① — 주문에 배송지를 붙인다

주마다 배송이 얼마나 다른지 보려면 주문마다 배송지가 붙어 있어야 한다. 배송지는 고객 표에 있으므로 주문 표에 붙인다. 붙이는 방식에 따라 행이 사라질 수 있으므로 세 줄짜리 표로 먼저 확인한다.

**5-1. 배송지를 찾지 못한 주문도 남겨 두고 두 표를 합쳐 보자.**

분석에서 빠질 주문을 직접 판단하려면 표를 붙이는 과정에서 주문이 사라지지 않아야 한다. 연결 번호가 A·B·C인 표와 B·C·D인 표를 붙일 때, 첫 표의 A도 남는지 확인한다.

`DataFrame`으로 두 표를 만든다. 왼쪽 표는 `key`에 A·B·C, `L`에 1·2·3을, 오른쪽 표는 `key`에 B·C·D, `R`에 20·30·40을 담는다. `merge`의 `on`에는 연결 열인 `key`를, `how`에는 왼쪽 행을 모두 남기는 left를 지정한다.

In [11]:
# 5-1. 두 표의 key는 각각 A·B·C와 B·C·D, L은 1·2·3, R은 20·30·40이다. 왼쪽 행을 남겨 key로 연결한다 (DataFrame, merge)
left = pd.DataFrame({"key": ["A", "B", "C"], "L": [1, 2, 3]})
right = pd.DataFrame({"key": ["B", "C", "D"], "R": [20, 30, 40]})
left.merge(right, on="key", how="left")

,key,L,R
0,A,1,NaN
1,B,2,20.0
2,C,3,30.0


> A·B·C 세 행이 다 남고, 짝이 없는 A의 `R`은 빈 값이다. 양쪽에 다 있는 키만 남기는 inner 방식이었다면 A는 사라진다.

주문 표를 왼쪽에 두고 `left`로 붙이면 주문이 사라지지 않는다.

**5-2. 실제 주문에 배송지를 붙이고 주문 수가 그대로인지 확인해 보자.**

주문 표를 왼쪽에 두고 고객 표를 `merge`로 연결한다. `on`에는 두 표에 공통으로 있는 `customer_id`, `how`에는 left를 지정해 결과를 `df`에 담는다. `len`으로 연결 전후의 행 수를 비교한다.

In [14]:
# 5-2. 주문 표에 고객 표를 연결해 df에 담고 전후 행 수를 비교한다. 연결 열은 customer_id, 방식은 left다 (merge, len)
customers['customer_id'].nunique(), orders['customer_id'].nunique()
df = orders.merge(customers, on="customer_id")
len(orders), len(df)

(99441, 99441)

> 둘 다 99,441행이다. 주문마다 `customer_state`가 붙었고 주문 수는 그대로다.

## 6. 계획 ② — 실제 소요일과 약속 소요일, 지금 약속의 지연율

약속일을 어떻게 바꿀지 정하려면 지금 약속이 실제보다 얼마나 여유가 있는지, 그런데도 어느 비율로 늦는지부터 알아야 한다. 기록이 2년치이므로 그동안 약속과 배송이 달라졌는지 보고 약속을 정할 기간을 고른다.

이 장은 영업일이 아니라 주말·공휴일을 포함한 달력상의 날짜 차이를 센다. 예상 배송일에는 날짜만 있으므로 약속한 날짜에 받았다면 시각이 늦어도 지연이 아니다.

**6-1. 늦은 밤 주문한 상품을 다음 날 새벽에 받았다면 소요일이 며칠인지 계산해 보자.**

9월 1일 23시에 주문해 9월 2일 1시에 받으면 경과 시간은 2시간이지만 달력상 날짜 차이는 1일이다. 고객에게 날짜로 약속하므로 이 장에서는 날짜 차이를 쓴다.

한 열의 값을 담는 `Series`에 두 시각을 넣는다. 이를 `pd.to_datetime`으로 날짜형으로 바꾸고, `dt.normalize`로 각 시각을 그날 자정에 맞춘다. `iloc`은 위치로 값을 고르며 0이 첫째, 1이 둘째 값이다. 수령일에서 구매일을 뺀 뒤 하루 길이를 나타내는 `pd.Timedelta`로 나누면 일 단위 소요일이 된다. 하루 길이는 `days`에 1을 지정한다.

In [16]:
# 6-1. 예시 구매·수령 시각을 날짜형으로 만들고 날짜를 맞춘 뒤 하루 단위 차이를 확인한다 (to_datetime, dt.normalize, Timedelta)
example_data = pd.Series(["2018-09-01 23:00:00", "2018-09-02 01:00:00"])
example_data = pd.to_datetime(example_data) # 다양한 날짜 양식을 자동으로 받아서 계산가능한 날짜객체로 바꿔주는 함수
example_data[1], example_data[0]
(example_data[1] - example_data[0]) / pd.Timedelta(days=1)


0.08333333333333333

**6-2. 실제 주문에서도 날짜 차이를 계산할 수 있도록 시각 열을 바꿔 보자.**

2-2에서 문자열로 읽힌 것을 확인했다. 구매 시각, 고객 수령 시각, 예상 배송일을 각각 `pd.to_datetime`으로 변환해 원래 열에 다시 담는다. `dtypes`로 세 열의 자료형이 바뀌었는지 확인한다.

In [ ]:
# 6-2. 세 열 각각을 pd.to_datetime으로 바꿔 되담는다. dtypes
# 'order_purchase_timestamp', 주문일
# 'order_delivered_customer_date', 도착일
# 'order_estimated_delivery_date', 도착예정일
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
orders.dtypes

> 세 열이 `datetime64`가 됐다.

**6-3. 배송에 며칠 걸렸는지 확인할 수 있는 주문을 골라 보자.**

2절에서 정한 대로 배송 완료 상태이면서 고객이 받은 시각이 있는 주문을 분석한다.

`order_status`가 delivered인 행을 고른 뒤 `dropna`로 수령 시각이 빈 행을 제외한다. `subset`에는 빈 값을 확인할 열 이름을 목록으로 지정한다. `copy`로 사본을 만들어 `d`에 담고, 제외 전 배송 완료 건수와 남은 행 수를 비교한다.

In [20]:
# 6-3. 배송 완료 상태이면서 수령 시각이 있는 행의 사본을 d에 담고 배송 완료 건수와 남은 행 수를 비교한다 (조건 필터, dropna, copy, sum, len)
mask = df['order_status'] == 'delivered' # 배송 상태가 도착인것만 골라서 마스킹
d = df[mask].dropna(subset=['oreder_delivered_customer_date']).copy() # 배송상태가 도착인것만 고른 데이터에서 결측치도 제거
d.shape

KeyError: ['oreder_delivered_customer_date']

> 배송 완료 96,478건 중 받은 시각이 빈 8건이 빠져 96,470건이다.

**6-4. 실제 배송에 걸린 날과 고객에게 약속한 날을 각각 계산해 보자.**

주문마다 두 소요일을 만들어야 약속이 실제와 얼마나 다른지 비교할 수 있다.

하루짜리 `pd.Timedelta`를 `DAY`에 둔다. 구매·수령 시각은 `dt.normalize`로 자정에 맞추고, 받은 날에서 구매한 날을 뺀 차이를 `DAY`로 나눠 `actual_days`에 담는다. 예상 배송일과 구매한 날의 차이도 일 단위로 바꿔 `promised_days`에 담는다. 두 열의 중앙값은 `median`으로 구한다.

In [21]:
# 6-4. 하루 길이를 DAY에 두고 날짜 차이를 일 단위로 바꿔 actual_days와 promised_days에 담는다. 각각의 중앙값을 본다 (Timedelta, dt.normalize, median)
DAY = pd.Timedelta(days=1) # 시간차이를 이용하기를 위한 보조변수
d['actual_day']=(d['order_delivered_customer_date'].dt.normalize() - d['order_purchase_timestamp'].dt.normalize()) / DAY
d['promised_days']=(d['order_estimeated_delivered_date'] - d['order_purchase_timestamp'].dt.normalize()) / DAY
d['actual_days'].median(), d['promised_days'].median()

NameError: name 'd' is not defined

> 실제 소요일 중앙값은 10일, 약속 소요일 중앙값은 24일이다.

두 분포의 중앙값 차이이며, 주문마다 14일의 여유가 있었다는 뜻은 아니다.

**6-5. 약속한 날짜를 넘겨 받은 주문이 얼마나 되는지 세어 보자.**

수령 시각을 `dt.normalize`로 날짜에 맞춘 뒤 예상 배송일보다 늦은지 비교해 `late`에 담는다. 참은 1, 거짓은 0으로 계산하므로 `mean`은 지연율, `sum`은 지연 건수다. 약속 당일에 받은 주문은 지연에 포함하지 않는다.

In [22]:
# 6-5. 주문별 수령일이 예상 배송일보다 늦은지를 late에 담고 비율과 건수를 센다 (dt.normalize, 비교, mean, sum)
mask = d['actual_days'] > d['promised_days'] # 배송에 걸린 기간이 배송약속 기간보다 기나?
d['late'] = mask
d['late'].mean(), d['late'].sum()

NameError: name 'd' is not defined

> 지연율 0.068, 6,534건이다.

**6-6. 배송이 대체로 비슷한 날에 끝나는지, 유난히 오래 걸린 주문도 있는지 살펴보자.**

중앙값만으로는 오래 걸린 주문이 얼마나 있는지 알 수 없다. 실제 소요일의 분포를 그리고 실제·약속 소요일의 중앙값을 함께 표시한다.

`plt.hist`에 `actual_days`를 넣어 히스토그램을 그린다. `clip`의 `upper`를 60으로 두어 그림에서만 60일 초과 값을 마지막 구간에 모은다. 원래 소요일은 바꾸지 않는다. `bins`는 구간 수이며 60으로 둔다. `axvline`으로 두 중앙값에 세로선을 긋고, `linestyle`로 점선, `color`로 색, `label`로 범례 이름을 지정한다. 축 이름과 제목을 붙이고 `legend`로 범례를 표시한다.

In [26]:
# 6-6. 크기 → actual_days를 60에서 clip한 히스토그램(bins 60) → 실제 중앙값·약속 중앙값에 axvline 점선(label) → 축 이름·제목·legend → 표시
import matplotlib.pyplot as plt
plt.hist(d['actual_days'], bins=60, clip=(upper=60)) # 실제 배송소요일의 분포
plt.axvline(d['actual_days'].median(), linestyle='--', color='r', label='actual') # 약속 배송일의 분포
plt.axvline(d['promised_days'].median(), linestyle='--', color='b', label='promised') # 중앙값을 수직선
plt.xlabel('days')
plt.title('Distribution of days')
plt.legend()
plt.show()

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (2773691934.py, line 3)

> 실제 소요일 분포는 오른쪽으로 길게 이어진다. 그림의 마지막 구간에는 60일을 넘긴 주문도 함께 들어 있다.

24일 선은 약속 소요일의 중앙값이다. 이 선을 넘었다고 모두 지연은 아니며, 지연은 각 주문에 실제 안내된 예상 배송일과 비교해 판정한다.

**6-7. 지역마다 주문 수와 지연율을 따로 계산해 보자.**

지역 A에 정시·지연 주문이 하나씩, B에 정시 주문 하나가 있으면 A는 2건에 지연율 0.5, B는 1건에 지연율 0이다. 이처럼 같은 지역의 주문끼리 모아야 지역별 값을 구할 수 있다.

지역 A·A·B와 지연 여부 거짓·참·거짓을 담은 표를 만든다. `groupby`에 지역 열을 지정하고, `agg`에는 새로 붙일 열 이름과 집계할 원래 열·함수를 지정한다. 원래 열 이름과 함수 이름은 괄호 안에 쉼표로 구분해 한 쌍으로 넣는다. 주문 수에는 `size`, 지연율에는 참·거짓의 평균인 `mean`을 쓴다.

In [28]:
# 6-7. 지역 A·A·B와 지연 여부 False·True·False인 예를 만들고 지역별 건수와 지연율을 구한다 (DataFrame, groupby, agg)
example_orders = pd.DataFrame({'region': ['A', 'A', 'B'],'late': [False, True, False],'price':[100,200,300]})
example_orders.groupby('region').agg(orders('late','size'),late_rate=('late','mean'),price('price','mean')) # 그룹별 집계함수

SyntaxError: positional argument follows keyword argument (3103300551.py, line 3)

**6-8. 구매한 달에 따라 약속 소요일과 지연율이 달라졌는지 비교해 보자.**

약속이나 배송이 달라졌다면 어느 기간의 주문으로 후보를 정할지도 달라진다. 월별 지연율과 함께 전체 지연 중 그 달이 차지하는 비율을 보면 지연이 몰린 달도 찾을 수 있다.

구매 시각을 `dt.to_period`로 월 단위로 바꿔 `month`에 담는다. 인자 M이 월 단위를 뜻한다. 이 열로 `groupby`한 뒤 `agg`로 주문 수·실제 소요일 중앙값·약속 소요일 평균·지연율·지연 건수를 집계해 `monthly`에 담는다. 월별 지연 건수를 전체 지연 건수로 나눠 `late_share`에 추가한다. `round`는 표시할 소수 자릿수를 정한다. 여기서는 3자리로 출력하며 원래 계산값은 그대로 둔다.

In [ ]:
# 6-8. 구매 월을 month에 담아 월별 주문 수·실제 소요일 중앙값·약속 평균·지연율·지연 건수를 monthly에 집계한다. late_share에는 전체 지연 중 그 달의 비율을 담는다 (dt.to_period, groupby, agg)
d['month'] = d['order_purchase_timestamp'].dt.to_period('M')
monthly = d.groupby('month').agg(orders=('order_id','size'),actual_days=('actual_days','median'),promised_days=('promised_days','mean'),late_rate=('late','mean'),late_count=('late','sum'))
monthly

> 약속 소요일 평균이 2017년 1월 40.2일에서 2018년 8월 15.8일로 짧아졌지만 중간에 다시 길어진 달도 있고, 실제 소요일 중앙값은 2018년 6월부터 7∼8일로 빨라졌다. 지연율은 대개 0.03∼0.07이지만 2017년 11월 0.124, 2018년 2월 0.141, 3월 0.190이다. 이 세 달에 전체 지연의 거의 반이 있다.

약속을 정할 기간을 여기서 정한다. 2년을 합치면 약속이 40일이던 때와 16일이던 때가 섞이고, 최근 몇 달만 쓰면 지연이 많았던 달이 빠진다. 이 장에서는 최근 12개월(2017년 9월∼2018년 8월)을 사용해 한 해 동안의 주문을 함께 비교한다. 이 표만으로 지연이 늘어난 원인이나 매년 같은 달에 반복되는지는 알 수 없다. 이 12개월 안에서도 2018년 7∼8월에 약속이 20.3일, 15.8일로 갑자기 짧아졌으므로, 회사가 7월부터 새 규칙을 쓰고 있다면 그 규칙과의 비교는 그 두 달로 따로 해야 한다.

**6-9. 선택한 최근 12개월에서 기존 약속과 지연율이 어느 수준인지 확인해 보자.**

앞에서 정한 2017년 9월∼2018년 8월의 주문을 조건 필터로 고른다. 배송 완료이며 수령 시각이 있는 표 `d`는 2018년 8월까지이므로, 구매 시각이 2017년 9월 1일 이상인 행의 사본을 `r`에 담는다. 주문 수는 `len`, 지연율과 약속 소요일 평균은 `mean`, 실제 소요일 중앙값은 `median`으로 구한다.

In [ ]:
# 6-9. 구매일이 2017년 9월 1일 이상인 주문의 사본을 r에 담고 건수·지연율·약속 평균·실제 중앙값을 확인한다 (조건 필터, copy, len, mean, median)
r=d[d["order_purchase"]]

> 74,206건, 지연율 0.078, 약속 소요일 평균 23.8일, 실제 소요일 중앙값 10일이다.

지연이 몰린 세 달이 다 들어 있어 2년 전체보다 지연율이 높다. 여기까지는 전국 전체의 값이다. 7절에서 주별로 나눠 지금 약속이 주마다 다른지, 실제와 맞는지 본다.

## 7. 계획 ③ — 주(state)마다 지금 약속과 실제가 얼마나 다른가

배송 소요일은 배송지에 따라 다르다. 주마다 주문 수, 실제 소요일 중앙값, 지금 약속 소요일, 지연율을 한 표에 놓으면 지금 약속이 주마다 다른지, 다르다면 실제와 맞는지가 보인다. 지연은 분포의 위쪽에서 생기므로 중앙값 대비 약속의 여유만으로는 지연율이 설명되지 않을 수 있다.

**7-1. 어느 주에서 배송 약속이 잘 지켜지지 않는지 비교해 보자.**

주마다 실제 배송에 걸리는 날과 안내한 날, 지연율을 함께 보면 약속이 긴데도 자주 늦는 지역을 찾을 수 있다. 약속 소요일의 표준편차도 구해 같은 주 안에서 주문별 약속이 얼마나 다른지 확인한다.

`r`을 배송지 열인 `customer_state`로 `groupby`하고, `agg`로 주문 수·실제 소요일 중앙값·약속 소요일 평균과 표준편차·지연율을 구한다. 표준편차에는 `std`를 쓴다. 실제 소요일 중앙값을 기준으로 `sort_values`로 정렬해 `by_state`에 담는다.

In [30]:
# 7-1. 주별 주문 수는 orders, 실제 중앙값은 median_days, 약속 평균과 표준편차는 promise_mean과 promise_sd, 지연율은 late_rate로 by_state에 집계하고 실제 중앙값순으로 정렬한다 (groupby, agg, sort_values)
by_state = r.groopyby('customer_state').agg(orders=('order_id','size'),median_days=('actual_days','median'),promise_mean=('promised_days','mean'),promise_sd=('promised_days','std'),late_rate=('late','mean'))
by_state.rond(3)

NameError: name 'r' is not defined

> SP(상파울루, 31,821건)는 실제 중앙값 7일에 약속 평균 19.1일, AM(아마조나스, 100건)은 28일에 49.6일이다. 지연율은 SP·PR 0.049, MG 0.055이고, RJ(리우데자네이루)는 실제 중앙값 13일에 약속 평균 27.2일인데도 지연율은 0.149다. BA 0.140, CE 0.167, MA 0.222이며 AM은 0.030, AP는 0이다.

지금 약속은 이미 주마다 다르고, 실제가 오래 걸리는 주일수록 약속도 길다. 약속 소요일의 표준편차는 어느 주나 6∼9일이라 같은 주 안에서도 주문마다 약속이 다르다. 주문이 1,000건 넘는 11개 주만 보면 지연율이 0.049에서 0.149로 세 배이고, 중앙값 대비 여유가 비슷한 주끼리도 지연율이 다르다. 그래서 8절에서는 중앙값이 아니라 분위수로 약속일을 정한다.

정할 것은 주마다 다르게 둘지가 아니라 주마다 며칠로 둘지다. 8절에서 주마다 실제 소요일 분포로 약속일을 정하는 규칙을 세운다.

## 8. 계획 ④ — 주(state)별 분위수로 약속일을 정한다

중앙값으로 약속하면 늦는 주문이 많을 수 있으므로 소요일 분포의 더 높은 위치를 살펴본다. 먼저 주별 90분위수를 후보로 두고, 여러 분위수에서 평균 약속일과 지연율을 비교한다. 그중 기존 전체 지연율 0.078과 가까운 후보를 고른다.

**8-1. 각 주문에 그 배송지의 약속 소요일을 붙여 보자.**

주별 후보를 평가하려면 각 주문이 어느 주의 약속을 적용받는지 알아야 한다. 배송지가 a·b·a이고 a의 약속은 10일, b는 20일이라면 주문별 약속은 10·20·10일이다.

배송지를 담은 `Series`와 지역 이름에 약속 소요일을 연결한 `Series`를 만든다. 배송지에 `map`을 적용하고 지역별 약속을 인자로 주면, 이름이 맞는 약속 소요일을 주문마다 찾아준다.

In [31]:
# 8-1. 값 a·b·a를 담은 Series에 a는 10, b는 20인 대응표를 적용한다 (Series, map)
pd.Series(['a', 'b', 'a']).map({'a': 10, 'b': 20})

,0
0,10
1,20
2,10


> 10, 20, 10이다. 각 값이 자기 이름에 해당하는 값으로 바뀌었다.

**8-2. 주문 5건 중 4건이 제때 도착하게 하려면 며칠을 약속해야 할지 생각해 보자.**

소요일이 1·2·3·4·5일이면 4일 약속에서 5건 중 4건이 제때 온다. 다만 데이터로 분위수를 계산한 값은 이 손 계산과 다를 수 있다.

소요일을 `Series`에 담고 `quantile`에 0.80을 지정한다. 기본 계산은 정렬한 값 사이를 비례에 따라 채우는 선형 보간이므로 결과는 4일이 아니라 4.2일이다. `np.ceil`은 소수 부분이 있으면 다음 정수로 올려 날짜로 안내할 수 있게 한다. 올림하면 약속을 지키는 주문이 달라질 수 있으므로 지연율도 다시 세어야 한다.

In [32]:
# 8-2. 소요일 1·2·3·4·5의 0.80분위수와 올림한 날짜를 확인한다 (Series, quantile, ceil)
import numpy as np
pd.Series([1, 2, 3, 4, 5]).quantile(0.80)
np.ceil(pd.Series([1, 2, 3, 4, 5]).quantile(0.80))

np.float64(5.0)

**8-3. 주별 90분위수를 약속하면 얼마나 길게 안내하고 얼마나 늦는지 확인해 보자.**

90분위수라고 해서 실제 지연율이 반드시 0.10인 것은 아니다. 약속과 같은 날에 받은 주문은 지연이 아니므로 이 비율도 함께 센다.

주별로 묶은 실제 소요일에 `quantile`을 적용해 `q90`에 담고, `map`으로 주문마다 대응시켜 `prom90`을 만든다. `mean`으로 평균 약속일을 구하고, 실제 소요일이 후보보다 긴지와 같은지를 각각 비교해 그 평균을 구한다. 여기서는 올림 전 분위수로 후보의 성질을 비교하고, 날짜로 안내할 후보는 9절에서 올림한다.

In [ ]:
# 8-3. 주별 실제 소요일 90분위수를 q90에 담고 주문별 후보 prom90에 대응시킨다. 평균 약속·지연 비율·같은 날 수령 비율을 확인한다 (groupby, quantile, map, mean)
q90 = r.grouby('customer_state')['actual_days'].quantile(0.9)
prom90 = r['actual_days'].map(q90)
(d['actual_days'] > prom90).mean(), (d['actual_days'] == prom90).mean()

> 평균 약속일은 22.5일, 지연율은 0.092다. 약속과 같은 날에 받은 비율은 0.013이다. 기존 약속보다 평균 1.3일 짧고 지연율은 0.014 높다.

같은 날에 받은 주문은 지연에 포함하지 않는다. 소요일에 같은 값이 반복되고 분위수 계산에 보간이 들어가므로 실제 지연율을 따로 세어야 한다.

**8-4. 기존 전체 지연율과 비슷하게 유지하려면 어느 분위수가 맞는지 비교해 보자.**

분위수를 높이면 약속이 길어지고 지연은 줄어든다. 후보 0.80·0.85·0.90·0.92·0.93·0.95에서 두 값이 함께 어떻게 바뀌는지 본다. 이 비교만으로 비용이 가장 낮은 분위수를 정할 수는 없다.

8-3의 계산을 `for`로 반복한다. 각 후보의 분위수, 평균 약속일, 지연율, 기존 평균 약속일과의 차이를 사전으로 만들고 `append`로 목록 `rows`에 모은다. 이를 `DataFrame`으로 바꿔 `policy`에 담으면 후보별 결과를 한 표에서 읽을 수 있다.

In [ ]:
# 8-4. 여섯 후보 p에서 주별 분위수 q와 주문별 약속 prom을 계산한다. p·mean_promise·late_rate·vs_current를 rows에 모아 policy로 만든다 (for, quantile, map, append, DataFrame)
row = []
for p in ()

> p가 0.80이면 평균 약속일 17.2일에 지연율 0.187, 0.95면 27.7일에 0.048이다. 비교한 후보 중 기존 지연율 0.078과 가장 가까운 것은 0.92로, 평균 약속일 24.1일에 지연율 0.075다. 지금 약속(23.8일, 0.078)과 평균 약속일도 지연율도 거의 같다.

p를 0.92로 두는 것은 기존 전체 지연율과 비슷한 수준을 유지한다는 3절의 조건을 따른 것이다. 전체 지연율을 비슷하게 두더라도 지역별 변화는 다르므로, 지연율이 낮던 주는 오르고 높던 주는 내려간다. 회사가 지연율을 더 낮추기로 하면 p를 올리면 되고, 그때 약속일이 얼마나 길어지는지는 이 표에 있다.

**8-5. 전국에 같은 소요일을 약속해도 지역별 지연율이 비슷할지 확인해 보자.**

주마다 약속을 다르게 둘 필요가 있는지 보려면 전국 공통 후보와 비교해야 한다. 앞에서 선택한 92분위수를 이번에는 전국 주문을 합쳐 구한다.

선택한 비율을 `P`, 전국 실제 소요일의 `quantile` 결과를 `national`에 담는다. 각 주문의 실제 소요일이 이 값보다 긴지 비교해 전체 평균을 구한다. 같은 참·거짓 값을 배송지로 `groupby`해 평균하면 주별 지연율이 된다. 이를 `sort_values`로 정렬해 `late_nat`에 담는다.

In [ ]:
# 8-5. 선택한 분위수를 P에 두고 전국 분위수 national을 구한다. 전국 공통 약속의 전체 지연율과 주별 지연율 late_nat를 비교한다 (quantile, 비교, groupby, mean, sort_values)

> 전국 92분위수는 26일이고 전체 지연율은 반올림하면 0.075로 주별 후보와 비슷하다. 그러나 주별로는 SP 0.020, MG 0.044이지만 RJ 0.150, BA 0.193, CE 0.242, PA 0.376, AM 0.560이다. 전체 지연율이 비슷해도 전국 하나의 값은 주마다 0.02에서 0.56까지 벌어진다.

지역별 지연율을 비슷한 수준으로 맞추려는 목적에는 전국 공통값이 맞지 않는다. 기존 약속에서도 지역별 차이가 있으므로, 주마다 92분위수로 다시 정하면 어느 주를 며칠 고칠지가 9절에서 나온다.

## 9. 계획 ⑤ — 주(state)별 조정표와 권고

주별로 기존 약속과 후보를 나란히 놓아 재검토할 지역을 찾는다. 기존 약속은 같은 주 안에서도 주문마다 다르지만, 후보는 주마다 하나의 소요일이다. 따라서 둘의 평균 차이를 기존 주문의 약속에 일괄적으로 더하는 방식과는 다르다. 후보를 적용했을 때의 지연율과 표본·기간에 따른 변화를 함께 읽는다.

**9-1. 어느 주의 약속을 며칠 앞당기거나 늘리는 후보인지 정리해 보자.**

기존 약속은 주문마다 다르므로 주별 평균으로 나타내고, 후보는 주마다 하나의 정수 소요일로 둔다. 조정 폭은 후보와 기존 평균의 차이다.

주별 92분위수를 `q_state`에 담고 `np.ceil`로 올림한다. 예를 들어 17.2일이면 18일이다. `DataFrame`에 주문 수, 실제 소요일 중앙값, 기존 약속 평균, 올림한 후보를 담아 `table`을 만든다. 주 이름이 같은 값끼리 한 행에 모인다. 후보에서 기존 평균을 뺀 `adjust_days`를 추가하고 이 값으로 정렬한다. 올림 후 지연율은 다음 셀에서 다시 센다.

In [ ]:
# 9-1. 주별 분위수 q_state를 구한다. table에 orders·actual_median·current_promise·올림한 recommended를 담고 후보와 기존 평균의 차이 adjust_days로 정렬한다 (value_counts, groupby, quantile, ceil, DataFrame, sort_values)

> SP(31,821건)는 19.1일에서 17일로 2.1일, MG는 24.8일에서 22일로 2.8일 앞당긴 후보이고, RJ(9,345건)는 27.2일에서 35일로 7.8일, BA는 4.9일, CE는 9.6일 늘린 후보이다. DF·GO·RS는 지금과 거의 같다.

주문이 상대적으로 적은 AP·AM·AC·RO·RR은 9-3에서 표본에 따른 변동을 확인한다.

**9-2. 새 약속을 적용하면 어느 주의 지연이 늘고 어느 주에서 줄어드는지 비교해 보자.**

전체 지연율이 비슷해도 각 지역의 변화는 다를 수 있다. 날짜로 올림한 후보를 실제 주문에 적용해 기존 약속과 나란히 본다.

`table`의 후보 소요일을 `map`으로 주문마다 붙여 `prom_new`에 담는다. 실제 소요일이 후보보다 긴지 비교한 뒤 주별로 평균해 `late_new`를 만든다. `DataFrame`에 주문 수, `by_state`의 기존 지연율, 새 지연율을 담고 기존 지연율 순으로 정렬한다.

In [ ]:
# 9-2. 주별 후보를 주문에 대응시켜 prom_new에 담고 주별 지연율 late_new를 구한다. orders·late_now·late_new를 나란히 비교한다 (map, 비교, groupby, mean, DataFrame)

> 과거 주문에 후보 약속을 적용하면 주별 지연율은 0.050∼0.085다. SP는 0.049에서 0.075로 올라 지연으로 분류되는 주문이 800건 넘게 늘고, RJ는 0.149에서 0.076으로 내려간다.

전체 지연율을 비슷하게 유지하는 것과 모든 주의 지연율을 낮추는 것은 다르다. 주별 결과가 비슷해진 것은 같은 주의 자료에서 같은 분위수를 골랐기 때문이기도 하다. 앞으로도 이 비율을 유지하는지는 별도 기간에서 확인해야 한다.

**9-3. 주문이 적은 지역의 약속 후보가 관측된 주문 구성에 얼마나 민감한지 확인해 보자.**

같은 주의 주문에서 원래와 같은 개수만큼 중복을 허용해 다시 뽑고 분위수를 구한다. 이 재표집을 500번 반복하면 관측된 주문 구성이 달라질 때 후보가 얼마나 변하는지 볼 수 있다. 주문이 적은 RR·AP·AC·AM·RO와 비교할 CE·SP를 살펴본다.

`default_rng`로 난수 생성기를 만들고 0을 지정해 결과를 다시 확인할 수 있게 한다. `choice`의 `size`에는 원래 주문 수, `replace`에는 중복을 허용하는 True를 둔다. 매번 `np.quantile`로 구한 값을 `qs`에 모은다. 그 값들의 0.05·0.95분위수가 `low`와 `high`이며, 미래 배송일의 범위가 아니다.

> **확장 실습.** 120분 수업에서는 교수자가 계산을 시연하고 학생은 범위를 해석한다. 반복문 작성은 수업 후에 한다.

In [ ]:
# 9-3. 난수 생성기를 고정하고 일곱 주에서 주문 수만큼 중복 허용 재표집을 500번 한다. 매번 P분위수를 qs에 모아 원자료 분위수와 재표집 분위수의 범위를 비교한다 (default_rng, choice, quantile, append, DataFrame)

> 재표집 분위수의 0.05∼0.95 범위는 SP 17∼17일, CE 37∼43일, AM 39∼41일이다. AP는 32∼41일, AC는 29∼51일, RR은 39∼71일이다.

AP·AC·RR의 조정 폭은 이 자료만으로 확정하기 어렵다. AM·RO의 범위가 상대적으로 좁아도 특정 표본 수 이상이면 안전하다는 기준이 되지는 않는다. 이 계산은 주 안의 관측 주문을 다시 뽑으므로, 아직 관측하지 못한 장기 지연이나 앞으로의 월별 변화까지 반영하지 못한다.

**9-4. 최근 석 달로 기간을 바꾸면 약속 조정 방향도 바뀌는지 비교해 보자.**

6-8에서 최근 배송이 빨라진 것을 확인했다. 지연이 몰렸던 달을 빼고 2018년 6∼8월로 계산했을 때도 같은 권고를 할 수 있는지 본다.

구매 시각이 2018년 6월 1일 이상인 주문을 `r3`에 담는다. 이 기간의 주별 분위수를 올림한 후보에서 같은 기간 기존 약속 평균을 빼 `adj3`에 담는다. `DataFrame`으로 12개월 값과 나란히 놓고 `loc`에 SP·MG·RJ·BA·CE의 목록을 지정해 해당 주의 행을 고른다.

In [ ]:
# 9-4. 2018년 6월 1일 이상 주문을 r3에 담는다. 주별 분위수를 올림한 후보와 같은 기간 기존 약속 평균의 차이를 adj3에 담고 SP·MG·RJ·BA·CE의 12개월 값과 비교한다 (조건 필터, groupby, quantile, ceil, DataFrame, loc)

> 최근 석 달만 쓰면 비교한 다섯 주 모두 앞당기는 후보가 나온다. RJ는 12개월 기준 7.8일 늘림에서 석 달 기준 10.4일 줄임으로 바뀐다.

2018년 6∼8월에는 실제 소요일이 짧아졌고, 지연이 많았던 달은 이 기간에 없다. 기간에 따라 실제 소요일과 기존 약속의 평균이 함께 바뀌므로 조정 방향도 달라진다. 이 비교는 서로 겹치는 기간에서 후보를 다시 계산한 것으로, 한 기간에서 정한 약속을 이후 주문에 적용한 검증은 아니다.

최근 12개월을 기준으로 보면 RJ·BA·CE·MA는 약속을 늘리고 SP·MG·PR은 앞당기는 후보가 나온다. 다만 기간을 바꾸면 방향이 뒤집히므로 이 값을 바로 운영에 적용하라고 권고하기는 어렵다. 우선 해당 지역의 최근 약속 규칙과 배송 변화를 확인하고, 앞선 기간에서 정한 후보를 이후 주문에 적용해 확인할 필요가 있다. 이 노트북은 그 검증 전의 조정 후보까지 제시한다.

### ✏️ 직접 해보기 — 조정 후보 설명하기 (8분)

SP와 RJ 중 한 주를 골라 기존 약속 평균, 후보 약속, 두 지연율을 앞의 표에서 찾아 적는다. 약속을 바꾸면 무엇이 좋아지고 무엇이 나빠지는지 설명하고, 12개월과 석 달의 조정 방향을 비교해 즉시 적용할 수 있는지 판단한다.

### 💬 생각해보기

- 같은 주 안에서 주문별 약속이 다른 이유를 확인하려면 어떤 정보가 필요한가.
- 지연율을 낮추는 것과 실제 배송을 빠르게 하는 것은 어떻게 다른가.
- 후보를 만든 주문에서 지연율이 낮았다는 사실만으로 다음 달의 결과를 기대해도 되는가.

## 10. 마무리

| 주제 | 핵심 |
|---|---|
| 데이터 | 주문 99,441건 중 배송 완료이며 수령 시각이 있는 96,470건을 사용한다. 고객 표를 붙인 뒤 주문 수를 확인한다 |
| 기간 | 최근 12개월의 74,206건을 비교한다. 기존 전체 지연율은 0.078이고 평균 약속은 23.8일이다 |
| 주별 차이 | 기존 지연율은 SP 0.049, RJ 0.149, MA 0.222다. 전체 평균만으로 지역별 상황을 알 수 없다 |
| 분위수 후보 | 비교한 후보 중 0.92에서 평균 약속 24.1일, 지연율 0.075로 기존 전체 지연율과 가깝다. 이 값은 올림 전 비교다 |
| 조정표 | 주별 92분위수를 올림한 후보는 RJ +8일, BA +5일, CE·MA +9∼10일, SP·MG·PR −2∼3일이다. 기존 주별 평균과의 차이며 개별 주문에 더할 값은 아니다 |
| 판단 | 과거 자료에서 만든 조정 후보다. 기간에 따라 방향이 달라지고 일부 주는 재표집 범위도 넓으므로 운영 적용 전 별도 기간에서 확인해야 한다 |

### 이번 주에 할 일

| 활동 | 자료 | 비고 |
|---|---|---|
| 배경지식 | 2주차 배경지식 문서 | 분포·평균과 분위수·서비스 수준·표본의 의미를 읽는다 |
| 교재 실습 | 이 노트북 | 계산을 채우고 선택한 주의 조정 후보를 설명한다 |
| 확장 실습 | 9-3 재표집 | 시연한 반복문을 직접 작성하고 범위를 해석한다 |
| 랩 | `week02_lab.ipynb` — Olist 확장 실습 | 앞선 주문으로 만든 주별 후보를 이후 주문에서 평가한다. 계산 6과제와 본인의 판단·설명 1과제, 약 60분 |